In [1]:
# 1. Tải mã nguồn gốc
!git clone https://github.com/pkuliyi2015/GeoBloom.git
%cd GeoBloom

# 2. Cài đặt các thư viện đặc thù (Ép phiên bản để tránh xung đột Python 3.12)
!pip install jieba_fast xxhash==3.4.1 scikit-learn==1.4.2

# 3. Biên dịch NNUE Engine tối ưu hóa phần cứng bằng g++
!g++ nnue/v19/nnue.cpp -o nnue/v19/nnue -pthread -mavx2 -O3 -fno-tree-vectorize

Cloning into 'GeoBloom'...
remote: Enumerating objects: 313, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 313 (delta 15), reused 20 (delta 6), pack-reused 276 (from 1)
Receiving objects: 100% (313/313), 196.66 MiB | 38.93 MiB/s, done.
Resolving deltas: 100% (145/145), done.
/kaggle/working/GeoBloom
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 47.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.2 MB/s eta 0:00:00
  Created wheel for jieba_fast: filename=jieba_fast-0.53-cp312-cp312-linux_x86_64.whl size=7659505 sha256=226faaa42b40cd729506d5bd14d9feb61b23a380be2718644025c4f3c3866aa5
  Stored in directory: /root/.cache/pip/wheels/65/67/37/8968a5b150cd26683fd229f2987694f1ff76c035f9ddc84bfe
Successfully built jieba_fast
  Attempting uninstall: xxhash
    Found e

In [2]:
# 1. Cài đặt p7zip và giải nén tập dữ liệu sạch GeoGLUE_clean
!apt-get update && apt-get install -y p7zip-full
!cd data && 7z x GeoGLUE_clean.7z -y

# 2. Chạy cấu trúc tiền xử lý dữ liệu sang file nhị phân .bin
!python model/dataset.py --dataset GeoGLUE_clean

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,646 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntu

In [3]:
import re

file_path = 'model/geobloom_v19.py'
with open(file_path, 'r') as f:
    content = f.read()

# Ép toàn bộ num_workers về 0 để bảo vệ tài nguyên RAM
new_content = re.sub(r'num_workers\s*=\s*\d+', 'num_workers=0', content)

with open(file_path, 'w') as f:
    f.write(new_content)
print("⚙️ Đã cấu hình tối ưu hệ thống: Ép num_workers về 0 thành công.")

⚙️ Đã cấu hình tối ưu hệ thống: Ép num_workers về 0 thành công.


In [4]:
import os
import time
import subprocess
import re

# Danh sách các tỷ lệ thiếu hụt dữ liệu huấn luyện do tác giả đề xuất
portions = [0.02, 0.05, 0.10, 0.20, 0.30, 0.50, 0.70]
dataset = 'GeoGLUE_clean'
epochs = 5 # Đặt số epoch huấn luyện tối ưu tĩnh như file gốc

log_output_path = 'varying_data_results.txt'

# Khởi tạo file log tĩnh mới
with open(log_output_path, 'w', encoding='utf-8') as log_file:
    log_file.write("=== KẾT QUẢ THỰC NGHIỆM VARYING TRAINING DATA PORTIONS ===\n")
    log_file.write("Portion\tRecall@20\tRecall@10\tNDCG@5\tNDCG@1\n")

print(f"🚀 Bắt đầu tiến trình thực nghiệm tự động trên {len(portions)} phân đoạn dữ liệu...")

for p in portions:
    print(f"\n=======================================================")
    print(f" 🔥 ĐANG XỬ LÝ TỶ LỆ DỮ LIỆU: {p * 100}% (Portion = {p})")
    print(f"=======================================================")
    
    # Bước 4.1: Tiến hành huấn luyện mô hình với tham số --portion
    print(f"-> 1. Đang huấn luyện sinh mô hình cho portion {p}...")
    train_cmd = f"python model/geobloom_v19.py --dataset {dataset} --portion {p} --epochs {epochs}"
    subprocess.run(train_cmd, shell=True, check=True)
    
    # Bước 4.2: Đánh giá mô hình bằng C++ NNUE Engine để lấy điểm số thực tế
    print(f"-> 2. Đang kích hoạt NNUE Engine kiểm thử tập test...")
    test_cmd = f"nnue/v19/nnue {dataset} test 8 800-800-800-800"
    process = subprocess.Popen(test_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    # Đọc kết quả đầu ra của C++ để nhặt điểm số
    recall_20, recall_10, ndcg_5, ndcg_1 = "-", "-", "-", "-"
    for line in process.stdout:
        # File C++ in kết quả theo cụm 4 số thập phân sau dòng Evaluation
        # Ta dùng regex quét dòng chứa các chỉ số float
        if re.match(r'^[0-9.]+\s+[0-9.]+\s+[0-9.]+\s+[0-9.]+', line.strip()):
            parts = line.strip().split()
            if len(parts) >= 4:
                recall_20 = parts[0]
                recall_10 = parts[1]
                ndcg_5 = parts[2]
                ndcg_1 = parts[3]
    process.wait()
    
    # Bước 4.3: Ghi nhận kết quả bóc tách được vào file kết quả tổng hợp
    with open(log_output_path, 'a', encoding='utf-8') as log_file:
        log_file.write(f"{p}\t{recall_20}\t{recall_10}\t{ndcg_5}\t{ndcg_1}\n")
        
    print(f"✅ Đã lưu kết quả portion {p}: Recall@20 = {recall_20}")

print(f"\n🎉 HOÀN THÀNH TOÀN BỘ TIẾN TRÌNH VARYING DATA! Kết quả lưu tại: {os.path.abspath(log_output_path)}")

🚀 Bắt đầu tiến trình thực nghiệm tự động trên 7 phân đoạn dữ liệu...

 🔥 ĐANG XỬ LÝ TỶ LỆ DỮ LIỆU: 2.0% (Portion = 0.02)
-> 1. Đang huấn luyện sinh mô hình cho portion 0.02...
[1/2] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output isin_cuda.cuda.o.d -DTORCH_EXTENSION_NAME=isin_cuda -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.12/dist-packages/torch/include -isystem /usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/cuda/include -isystem /usr/include/python3.12 -D__CUDA_NO_HALF_OPERATORS__ -D__CUDA_NO_HALF_CONVERSIONS__ -D__CUDA_NO_BFLOAT16_CONVERSIONS__ -D__CUDA_NO_HALF2_OPERATORS__ --expt-relaxed-constexpr -gencode=arch=compute_75,code=compute_75 -gencode=arch=compute_75,code=sm_75 --compiler-options '-fPIC' -lineinfo -std=c++17 -c /kaggle/working/GeoBloom/cuda/isin_cuda.cu -o isin_cuda.cuda.o 
[2/2] c++ isin_cuda.cuda.o -shared -L/usr/local/lib/python3.12/dist-packages/torch/lib -lc10 -l

/kaggle/working/GeoBloom/model/geobloom_v19.py:467: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast()
/kaggle/working/GeoBloom/model/geobloom_v19.py:589: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast()
Reading Bloom filters: 100%|██████████| 777295/777295 [00:05<00:00, 152862.87it/s]


Deserializing data_bin/GeoGLUE_clean/poi.bin takes 7.1172568798065186 seconds.


Loading query data:   0%|          | 0/600 [00:00<?, ?it/s]Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.777 seconds.
Prefix dict has been built succesfully.
Reading Bloom filters:   0%|          | 0/12157 [00:00<?, ?it/s]

Serializing data_bin/GeoGLUE_clean/portion/train_0.02.bin takes 0.001512289047241211 seconds.
Deserializing data_bin/GeoGLUE_clean/dev.bin takes 0.07008004188537598 seconds.


...

==================== Epoch 4 ====================
Train recall: [np.float64(0.9596348595064899), np.float64(0.9350068939285884), np.float64(0.9210288594114011), np.float64(0.9045785194694053)], Train NDCG @ 5: 0.5613767625458369
Dev recall: [np.float64(0.9828855040219066), np.float64(0.9549033030977238), np.float64(0.9221290432996748), np.float64(0.8782303611158652)], Dev NDCG @ 5: 0.5412174374584406
Previous max metrics: [np.float64(0.9822864966626733), np.float64(0.9560157453362998), np.float64(0.9272633920931028), np.float64(0.8871298990244737), np.float64(0.5417598088003648)]
Current max metrics: [np.float64(0.9828855040219066), np.float64(0.9560157453362998), np.float64(0.9272633920931028), np.float64(0.8871298990244737), np.float64(0.5417598088003648)]
Preparing training data and targets...


Encoding nodes:  50%|█████     | 2/4 [00:00<00:00, 12.27it/s]

Epoch=4, retrieve loss=1637.6559691578905, rank loss=2616.0868461632076
Max metrics: [np.float64(0.9828855040219066), np.float64(0.9560157453362998), np.float64(0.9272633920931028), np.float64(0.8871298990244737), np.float64(0.5417598088003648)]


Encoding nodes: 100%|██████████| 4/4 [00:01<00:00,  3.35it/s]


Beam width: 800 800 800 800 Loading dataset from path: data_bin/GeoGLUE_clean/
Allocating 277.549 MB for 142105 bloom filters on the tree...
Allocating 94.8846 MB for 777295 leaf nodes at depth 0...
Infering test candidates...
Searching 10000th query...
Total search time of all threads: 105.03s, Query Per Second: 115.748
=============== Intermediate Recall Scores ==============
0.986839	0.959941	0.928272	0.889282	
====================== Evaluation =======================
Recall@20 	 Recall@10  	 NDCG@5  	 NDCG@1
0.782841	0.726413	0.546229	0.416961
Predictions saved to data_bin/GeoGLUE_clean/test_nodes.bin
Node representations saved to data_bin/GeoGLUE_clean/node_v19.bin
Model serialized to data_bin/GeoGLUE_clean/nnue_v19.bin
Best dev time: 2918.199009180069 (s)
-> 2. Đang kích hoạt NNUE Engine kiểm thử tập test...
✅ Đã lưu kết quả portion 0.7: Recall@20 = 0.782841

🎉 HOÀN THÀNH TOÀN BỘ TIẾN TRÌNH VARYING DATA! Kết quả lưu tại: /kaggle/working/GeoBloom/varying_data_results.txt
